In [5]:
import os


In [6]:
%pwd

'd:\\Text-Summerizer-project\\research'

In [7]:
os.chdir("../")

In [8]:
%pwd

'd:\\Text-Summerizer-project'

In [9]:
import os

os.environ["HF_HOME"] = "D:/ml_project/hf_cache"

class ConfigurationManager:
    def __init__(self):
        self.root_dir = "D:/ml_project/artifacts"
        self.data_path = "D:/ml_project/datasets"

In [10]:
print(os.environ["HF_HOME"])

D:/ml_project/hf_cache


In [11]:
import os

os.environ["HF_HOME"] = "D:/ml_project/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "D:/ml_project/hf_cache"
os.environ["HF_DATASETS_CACHE"] = "D:/ml_project/hf_cache"
os.environ["TORCH_HOME"] = "D:/ml_project/hf_cache"

In [12]:
import transformers
print(transformers.__version__)

d:\Text-Summerizer-project\textS\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


4.41.0


d:\Text-Summerizer-project\textS\lib\site-packages\transformers\utils\hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [13]:
import os
print(os.getcwd())

d:\Text-Summerizer-project


In [14]:
import os

os.chdir(r"D:\Text-Summerizer-project")
print(os.getcwd())

D:\Text-Summerizer-project


In [15]:
import os

print("Current directory:", os.getcwd())

for root, dirs, files in os.walk("."):
    if "config.yaml" in files:
        print("Found:", os.path.join(root, "config.yaml"))

Current directory: D:\Text-Summerizer-project
Found: .\config\config.yaml


In [16]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: Path
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    evaluation_strategy: str
    eval_steps: int
    save_steps: float
    gradient_accumulation_steps: int 

In [17]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml,create_directories

In [18]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        
        create_directories([self.config.artifacts_root])
    
    
    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.TrainingArguments   # NOTE: params should come from self.params

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
             root_dir=config.root_dir,
             data_path=config.data_path,
            model_ckpt=config.model_ckpt,

            num_train_epochs=params.num_train_epochs,
            warmup_steps=params.warmup_steps,
            per_device_train_batch_size=params.per_device_train_batch_size,
            weight_decay=params.weight_decay,
            logging_steps=params.logging_steps,
            evaluation_strategy=params.evaluation_strategy,
            eval_steps=params.eval_steps,
            save_steps=params.save_steps,
            gradient_accumulation_steps=params.gradient_accumulation_steps
        )   

        return model_trainer_config
        

In [19]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
import torch

W0605 15:37:33.848831 11232 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


[2026-06-05 15:37:36,277:INFO:datasets:PyTorch version 2.11.0+cu128 available.]


In [20]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"

        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)

        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(
            self.config.model_ckpt
        ).to(device)

        seq2seq_data_collector = DataCollatorForSeq2Seq(
            tokenizer, model=model_pegasus
        )

        dataset_samsum_pt = load_from_disk(self.config.data_path)

        trainer_args = TrainingArguments(
            output_dir=self.config.root_dir,
            num_train_epochs=1,
            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=4,
            logging_steps=10,
            save_steps=1000,
            eval_strategy="no",
            report_to="none"
        )

        trainer = Trainer(
            model=model_pegasus,
            args=trainer_args,
            tokenizer=tokenizer,
            data_collator=seq2seq_data_collector,
            train_dataset=dataset_samsum_pt["train"],
            eval_dataset=dataset_samsum_pt["validation"]
        )

        trainer.train()

        os.makedirs(self.config.root_dir, exist_ok=True)

        model_pegasus.save_pretrained(
            os.path.join(self.config.root_dir, "flan-t5-small-samsum-model")
        )

        tokenizer.save_pretrained(
            os.path.join(self.config.root_dir, "tokenizer")
        )

In [21]:
from datasets import load_from_disk

dataset_samsum_pt = load_from_disk(
    r"D:\Text-Summerizer-project\artifacts\data_transformation\samsum_dataset"
)

print(dataset_samsum_pt)

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 14732
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 819
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 818
    })
})


In [22]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer_config = ModelTrainer(config=model_trainer_config)
    model_trainer_config.train()
except Exception as e:
    raise e

[2026-06-05 15:37:37,214:INFO:textSummarizerLogger:yaml file: config\config.yaml loaded successfully]
[2026-06-05 15:37:37,222:INFO:textSummarizerLogger:yaml file: params.yaml loaded successfully]
[2026-06-05 15:37:37,223:INFO:textSummarizerLogger:created directory at: artifacts]
[2026-06-05 15:37:37,224:INFO:textSummarizerLogger:created directory at: artifacts/model_trainer]


  0%|          | 10/3683 [00:06<36:43,  1.67it/s] 

{'loss': 37.3209, 'grad_norm': 93.6996841430664, 'learning_rate': 4.986424110779257e-05, 'epoch': 0.0}


  1%|          | 20/3683 [00:12<35:38,  1.71it/s]

{'loss': 30.4732, 'grad_norm': 65.1352310180664, 'learning_rate': 4.972848221558512e-05, 'epoch': 0.01}


...

{'loss': 0.4691, 'grad_norm': 1.3015468120574951, 'learning_rate': 1.7648655986967146e-07, 'epoch': 1.0}


100%|█████████▉| 3680/3683 [33:42<00:01,  1.87it/s]

{'loss': 0.412, 'grad_norm': 0.9698905944824219, 'learning_rate': 4.072766766223188e-08, 'epoch': 1.0}


100%|██████████| 3683/3683 [33:44<00:00,  1.82it/s]


{'train_runtime': 2024.7953, 'train_samples_per_second': 7.276, 'train_steps_per_second': 1.819, 'train_loss': 0.9049594712393328, 'epoch': 1.0}
